# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIRˆ2 dataset using the `mlcroissant` library. All field and entity references use their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is publicly available via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Dataset version: {getattr(metadata, 'version', '(unknown)')}")


## 2. Data Overview
Review and print available record sets, fields, and their IDs from the Croissant metadata. This will help us identify which record sets and fields (columns) to extract or analyze. All references use their `@id` for clarity.

In [ ]:
# List available record sets
record_sets = list(dataset.record_sets)
print("Available record set @ids and names:")
for rs in record_sets:
    print(f"  @id: {rs['@id']}  |  name: {rs.get('name', '')}")

# Let's inspect the first record set's fields:
if record_sets:
    example_rs_id = record_sets[0]['@id']
    fields = dataset.fields(record_set=example_rs_id)
    print(f"\nFields (columns) in record set '@id={example_rs_id}':")
    for field in fields:
        print(f"  @id: {field['@id']}  |  name: {field.get('name', '')}  |  type: {field.get('dataType', '')}")
else:
    print('No record sets found in the dataset.')

## 3. Data Extraction
Extract all available record sets by their `@id` and load as pandas DataFrames for analysis. Use the list of record set IDs and refer to columns by `@id`.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set '@id={record_set_id}' loaded: {df.shape[0]} records, {df.shape[1]} fields")

# Display columns of the first available record set for reference
if dataframes:
    display_rs_id = record_set_ids[0]
    print(f"\nColumns (@id) in '@id={display_rs_id}':")
    print(dataframes[display_rs_id].columns.tolist())
    dataframes[display_rs_id].head()
else:
    print('No dataframes could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing, filtering by field values, normalizing a numeric column, and grouping data using the dataset's field and record set `@id`s. (Adapt field `@id` names to your use case as discovered above!)

In [ ]:
# Example EDA: select a numeric field by its @id from the main record set
# You may print available columns in section 3 to identify the correct @ids.

# Replace with actual record_set_id and field @ids found above, e.g.:
main_rs_id = record_set_ids[0] if record_set_ids else None
if not main_rs_id:
    raise ValueError("No record set available for EDA.")

# We'll make these up for illustration, please update as appropriate for this dataset:
# e.g. '@id' for age: 'https://api.app.sen.science/frontiers/7862866/field-age'
# For demonstration purpose, we try to infer possible numeric fields
possible_numeric = []
for col in dataframes[main_rs_id].columns:
    # crude heuristic: look for common field names in clinical data
    if any(key in col.lower() for key in ['age', 'interval', 'metastasis', 'stage']):
        possible_numeric.append(col)
numeric_field_id = possible_numeric[0] if possible_numeric else dataframes[main_rs_id].columns[0]
print(f"Using numeric field @id: {numeric_field_id}")

# We will use a threshold for EDA filtering (update as suitable to the field)
df = dataframes[main_rs_id].copy()
try:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
except Exception as e:
    print(f"Could not convert {numeric_field_id} to numeric: {e}")
    
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df.copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {filtered_df.shape[0]} rows")
print(filtered_df.head())

# Normalize if numeric
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized column '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Field '{numeric_field_id}' could not be normalized (not numeric).")

# Attempt to group by a categorical field
group_field_id = None
categorical_candidates = [col for col in df.columns if col != numeric_field_id]
if categorical_candidates:
    group_field_id = categorical_candidates[0]

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    grouped_df.columns = [group_field_id, f"avg_{numeric_field_id}"]
    print(f"\nGrouped average of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())


## 5. Visualization
Visualize the distribution of the selected numeric field and group averages, if possible. Modify below to use the extracted fields' `@id` for all axes and labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of numeric field (@id={numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If we created a grouped average table, plot it
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(8,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=f"avg_{numeric_field_id}")
    plt.title(f"Average of '{numeric_field_id}' grouped by '{group_field_id}' (@id)")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Average {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to explore a Croissant-packaged clinical dataset. We:
- Loaded the FAIRˆ2 dataset and displayed its metadata,
- Enumerated record sets and their fields using `@id`,
- Extracted all record sets as DataFrames,
- Performed basic EDA steps (filtering, normalization, grouping),
- Visualized numeric field distributions and group averages using field `@id`s.

**Tip:** Always refer to dataset entities (`recordSet`, `field`, etc.) by their stable `@id` when working reproducibly with Croissant datasets and `mlcroissant`. You can now proceed to more advanced modeling or reporting using this tabular data. For details on field meanings or to adapt this notebook, return to the Croissant schema documentation.